# StratoRace: Exploratory Data Analysis (EDA)

This notebook visualises the raw F1 telemetry from FastF1 (`lap_telemetry.parquet`) to understand tyre degradation, compound choices, and pit stop windows across real F1 races (2022-2024).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#0a1628", "figure.facecolor": "#060c1a", 
                                    "text.color": "#7a9ab5", "axes.labelcolor": "#7a9ab5",
                                    "xtick.color": "#7a9ab5", "ytick.color": "#7a9ab5"})
palette = {'SOFT': '#E8002D', 'MEDIUM': '#FFD600', 'HARD': '#FFFFFF', 'INTER': '#00c850', 'WET': '#00aaff'}

In [ ]:
# Load Telemetry Data
data_path = "stratorace/backend/data/lap_telemetry.parquet"
if os.path.exists(data_path):
    df = pd.read_parquet(data_path)
    print(f"Loaded {len(df)} laps of real telemetry.")
else:
    print(f"File not found: {data_path}. Run ingest_fastf1.py first.")
    df = pd.DataFrame() # dummy

## 1. Tyre Degradation: Lap Time Delta vs Tyre Age (Scatter/LM)

In [ ]:
if not df.empty:
    valid_laps = df[
        (df['compound'].isin(['SOFT', 'MEDIUM', 'HARD'])) &
        (~df['is_pit_in']) & (~df['is_pit_out']) &
        (df['lap_time_delta'].abs() < 5.0) &
        (df['tyre_life'] > 0)
    ]
    
    plt.figure(figsize=(12, 6))
    sns.lmplot(x='tyre_life', y='lap_time_delta', hue='compound', data=valid_laps, 
               palette=palette, scatter_kws={'alpha':0.1, 's':10}, 
               line_kws={'linewidth':3}, height=6, aspect=1.5, x_jitter=0.5)
    
    plt.title('Tyre Degradation: Lap Time Delta by Compound', color='white', fontsize=16)
    plt.xlabel('Tyre Age (Laps)')
    plt.ylabel('Lap Time Delta to Race Average (s)')
    plt.ylim(-3, 4)
    plt.show()

## 2. Violin Plot: Lap Time Delta Distribution by Compound

In [ ]:
if not df.empty:
    plt.figure(figsize=(12, 6))
    sns.violinplot(x="compound", y="lap_time_delta", data=valid_laps, palette=palette, inner="quartile")
    plt.title('Distribution of Lap Time Deltas by Compound', color='white', fontsize=16)
    plt.xlabel('Compound')
    plt.ylabel('Lap Time Delta (s)')
    plt.ylim(-3, 5)
    plt.show()

## 3. Pit Stop Windows (Stint Length Distribution)

In [ ]:
if not df.empty:
    pit_laps = df[df['is_pit_in'] & df['compound'].isin(['SOFT', 'MEDIUM', 'HARD'])]
    
    plt.figure(figsize=(12, 6))
    sns.histplot(data=pit_laps, x='tyre_life', hue='compound', palette=palette, 
                 multiple='stack', bins=range(0, 55, 2))
    
    plt.title('Distribution of Stint Lengths (Tyre Age at Pit Stop)', color='white', fontsize=16)
    plt.xlabel('Tyre Age (Laps)')
    plt.ylabel('Number of Pit Stops')
    plt.show()

## 4. Position vs. Gap Ahead (Boxplot)

In [ ]:
if not df.empty:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='position', y='gap_ahead', data=df[df['gap_ahead'] < 20], color='#B8FF00')
    plt.title('Gap Ahead by Race Position', color='white', fontsize=16)
    plt.xlabel('Race Position')
    plt.ylabel('Gap to Car Ahead (s)')
    plt.show()

## 5. Scatter Plot: Speed Trap vs Track Temperature

In [ ]:
if not df.empty:
    sample_df = df[(df['speed_st'].notna()) & (df['track_temp'].notna())].sample(min(10000, len(df)))
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='track_temp', y='speed_st', data=sample_df, alpha=0.3, color='#00aaff', edgecolor=None)
    plt.title('Speed Trap vs Track Temperature', color='white', fontsize=16)
    plt.xlabel('Track Temperature (°C)')
    plt.ylabel('Speed Trap (km/h)')
    plt.show()

## 6. Correlation Heatmap of Continuous Telemetry Variables

In [ ]:
if not df.empty:
    cols = ['tyre_life', 'lap_time_delta', 'gap_ahead', 'gap_behind', 'track_temp', 'air_temp', 'position', 'speed_st']
    corr = df[cols].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, cbar_kws={"shrink": .8})
    plt.title('Correlation Matrix of Telemetry Features', color='white', fontsize=16)
    plt.show()

## 7. Confusion Matrix: Pit Strategy Compound Transitions

In [ ]:
if not df.empty:
    # Create a DataFrame of compound transitions during pit stops
    transitions = []
    for (year, gp, driver), group in df.groupby(['year', 'gp', 'driver']):
        group = group.sort_values('lap_number')
        prev_compound = group.iloc[0].get('compound', 'UNKNOWN')
        for idx, row in group.iterrows():
            if row['is_pit_in']:
                current_compound = row.get('compound', 'UNKNOWN')
                if prev_compound != current_compound:
                    transitions.append({'From': prev_compound, 'To': current_compound})
                prev_compound = current_compound
                
    trans_df = pd.DataFrame(transitions)
    if not trans_df.empty:
        cm = pd.crosstab(trans_df['From'], trans_df['To'])
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title('Confusion Matrix: Tyre Compound Transitions', color='white', fontsize=16)
        plt.xlabel('Switched To')
        plt.ylabel('Switched From')
        plt.show()

## 8. JointPlot (Scatter + Density): Gap Ahead vs Gap Behind

In [ ]:
if not df.empty:
    sample_gaps = df[(df['gap_ahead'] < 30) & (df['gap_behind'] < 30)].sample(min(5000, len(df)))
    g = sns.jointplot(x='gap_ahead', y='gap_behind', data=sample_gaps, 
                  kind="hex", color="#B8FF00", height=8)
    g.fig.suptitle('Density: Gap Ahead vs Gap Behind', color='white', fontsize=16, y=1.02)
    g.set_axis_labels('Gap Ahead (s)', 'Gap Behind (s)', color='#7a9ab5')
    plt.show()